# Week 3 – Scores, loss functions, and how a model learns (backprop)

There were two Week 3 labs:
* **Loss Function lab** (`loss_function_unsolved.ipynb`): compute model scores with $Wx + b$, the SVM hinge loss, softmax probabilities, and the $-\ln P$ loss.
* **Backprop lab**: structure a notebook into the main parts of a machine learning pipeline.

This notebook does both. It's the most important week: **Week 4 PyTorch is just this week, automated.**

## 1. From an image to scores: $s = Wx + b$
A classifier takes an input and gives one **score** per class. Higher score = "I think it's this class".

* $x$ – the input as a column of numbers. For MNIST you'd flatten the 28×28 image into 784 numbers. In the lab, $x$ has 10 numbers.
* $W$ – the **weight matrix**: one row per class, one column per input number. 5 classes × 10 inputs → shape (5, 10).
* $b$ – the **bias**: one extra number per class, shape (5, 1).
* $s = Wx + b$ – the **scores**, one per class, shape (5, 1).

**Each score is just a weighted sum of the inputs.** Row 1 of $W$ multiplied by $x$, plus $b_1$, gives the score for class 1. Training = finding a good $W$ and $b$.

### Matrix multiplication by hand
For a matrix times a column: each output = (row of the matrix) · (column), i.e. multiply pairs and add up.

In [ ]:
import numpy as np

W = np.array([[1, 2, 3],
              [0, 1, 0]])        # shape (2, 3): 2 classes, 3 inputs
x = np.array([[1],
              [1],
              [2]])              # shape (3, 1): a column
# class 0 score: 1*1 + 2*1 + 3*2 = 9
# class 1 score: 0*1 + 1*1 + 0*2 = 1
print(np.dot(W, x))              # same as W @ x

**Shape rule:** `(a, b) @ (b, c) → (a, c)`. The inner numbers must match. (2, 3) @ (3, 1) → (2, 1). ✅
If you get `shapes not aligned`, print the shapes – an inner number doesn't match.

### Lab Task 1 & 2
Random $x$ (10 numbers), $W$ (5 × 10) and $b$ (5) from a uniform distribution on [0, 1], then the scores.

In [ ]:
np.random.seed(42)            # so your numbers match mine
inp_dim = 10
num_classes = 5

x = np.random.rand(inp_dim)                 # shape (10,)
w = np.random.rand(num_classes, inp_dim)    # shape (5, 10)
b = np.random.rand(num_classes)             # shape (5,)

# The lab turns x and b into column vectors. [:, None] adds a second dimension of size 1.
x = x[:, None]       # (10,) -> (10, 1)
b = b[:, None]       # (5,)  -> (5, 1)
print("x", x.shape, "| w", w.shape, "| b", b.shape)

scores = np.dot(w, x) + b                   # (5,10) @ (10,1) + (5,1) -> (5,1)
print("scores shape:", scores.shape)
print(scores.ravel())                        # .ravel() flattens to 1-D just for printing

## 2. Loss functions: one number that says "how wrong"
Scores alone don't tell us if the model is good. We need the **correct class** (the label) and a **loss**: a single number that is **small when the model is right and confident**, and **big when it's wrong**. Training = making the loss small.

Throughout, suppose the correct class is **class 1** = index **0** in Python.

### 2.1 SVM hinge loss (Lab Task 3)
Idea: the correct class's score should beat every other class's score **by at least 1** (a "margin"). Each wrong class that gets too close adds a penalty:
$$L = \sum_{j \neq y} \max\big(0,\ s_j - s_y + 1\big)$$
* $s_y$ – score of the correct class. $s_j$ – score of a wrong class.
* If $s_j$ is more than 1 below $s_y$: $s_j - s_y + 1 < 0$, so `max(0, ...)` gives **0 penalty**. 👍
* Otherwise the penalty is how far it is inside the margin.

In [ ]:
correct = 0                                   # class 1 -> index 0
margins = np.maximum(0, scores[1:] - scores[correct] + 1)   # scores[1:] = every class except index 0
print("penalty from each wrong class:", margins.ravel())

hinge_loss = np.sum(margins)
print("hinge loss:", hinge_loss)

> **Note:** the lab's solution stops at the `np.maximum(...)` line, which shows the *per-class* penalties. The actual loss is their **sum** (the second line). `scores[1:]` only works because the correct class is index 0 – the general version is below.

In [ ]:
def hinge(scores, correct):
    s = scores.ravel()
    margins = np.maximum(0, s - s[correct] + 1)
    margins[correct] = 0            # don't compare the correct class with itself
    return np.sum(margins)

print(hinge(scores, 0))   # same as above
print(hinge(scores, 3))   # if the correct class were class 4 instead

### 2.2 Softmax: turning scores into probabilities (Lab Task 4)
Scores can be any number (negative, huge...). We'd like **probabilities**: all between 0 and 1 and adding up to 1. Softmax does that in two steps:
1. $e^{s_j}$ – exponentiate every score (makes them all positive; bigger score → much bigger number).
2. Divide by the total so they add to 1.
$$P(\text{class } j) = \frac{e^{s_j}}{\sum_k e^{s_k}}$$

In [ ]:
prob = np.exp(scores) / np.sum(np.exp(scores))
print("probabilities:", np.round(prob.ravel(), 3))
print("they sum to:", prob.sum())
print("model's guess = class index", np.argmax(prob))    # argmax = position of the biggest value

### 2.3 Cross-entropy / negative log-likelihood (Lab Task 5)
$$L = -\ln P(Y = y_i \mid X = x_i)$$
In words: **take the probability the model gave to the correct class, and take minus its log.** That's it.

In [ ]:
loss = -np.log(prob[correct]).item()     # .item() pulls the single number out of the array
print("loss:", loss)

# Why -ln? Look at how it behaves:
for p in [1.0, 0.9, 0.5, 0.1, 0.01]:
    print(f"P(correct) = {p:<5} -> loss = {0 - np.log(p):.3f}")

* Model 100% sure and right → loss 0.
* Model gives the right answer only 1% → loss 4.6 (big).
* The log punishes confident wrong answers *very* hard.

**Remember this number:** with 10 classes, a model that's just guessing gives each class probability 0.1, so loss = $-\ln(0.1) \approx 2.30$. In Week 4 your first-epoch MNIST loss starts around 2.2–2.3 – that's the network starting from random guessing.

This is exactly what PyTorch's `nn.CrossEntropyLoss` computes (softmax + $-\ln$ in one go).

### Try it
Assume the correct class is class 3 (index 2). Compute the hinge loss and the cross-entropy loss using the same `scores`.

In [ ]:
# your code here

In [ ]:
# ---- Answer ----
print("hinge:", hinge(scores, 2))
print("cross-entropy:", -np.log(prob[2]).item())

## 3. How does the model learn? Gradient descent
We have a loss. We want to change the weights to make it smaller. The **gradient** (derivative) tells us, for each weight, *which direction makes the loss go up*. So we step the **opposite** way:
$$w \leftarrow w - \text{learning rate} \times \frac{\partial L}{\partial w}$$

### A one-weight example
Loss $L(w) = (w - 3)^2$. The best $w$ is obviously 3. The derivative is $\frac{dL}{dw} = 2(w - 3)$.

In [ ]:
import matplotlib.pyplot as plt

w = -4.0               # start somewhere bad
lr = 0.1               # learning rate: how big each step is
history = []
for step in range(30):
    grad = 2 * (w - 3)         # derivative of the loss at the current w
    w = w - lr * grad          # step downhill
    history.append((w - 3) ** 2)

print("final w:", round(w, 4))
plt.plot(history); plt.xlabel("step"); plt.ylabel("loss"); plt.title("loss going down"); plt.grid(); plt.show()

**Try it:** set `lr = 1.1` and re-run. The loss explodes – too big a step overshoots the bottom every time. Set `lr = 0.001` – it barely moves. Choosing the learning rate matters (in Week 4, `lr=1e-3` is why accuracy climbs slowly).

## 4. Backpropagation = the chain rule, done automatically
In a real network the loss depends on the weights through several steps (weights → scores → softmax → loss). To get the gradient we use the **chain rule**: multiply the derivatives of each step together, working **backwards** from the loss. That's all "backpropagation" means.

Small example: prediction $p = w \cdot x$, loss $L = (p - t)^2$.
$$\frac{dL}{dw} = \underbrace{\frac{dL}{dp}}_{2(p-t)} \times \underbrace{\frac{dp}{dw}}_{x}$$

In [ ]:
x_val, t, w_val = 2.0, 10.0, 3.0
p = w_val * x_val                  # forward pass: p = 6
L = (p - t) ** 2                   # loss = 16
dL_dp = 2 * (p - t)                # -8
dp_dw = x_val                      # 2
print("by hand: dL/dw =", dL_dp * dp_dw)      # -16

PyTorch does this for you. Mark a tensor with `requires_grad=True`, compute the loss, call `.backward()`, and the gradient appears in `.grad`. **This is what `loss.backward()` does in Week 4.**

In [ ]:
import torch

w_t = torch.tensor(3.0, requires_grad=True)   # 'track everything done to w'
p_t = w_t * 2.0
L_t = (p_t - 10.0) ** 2
L_t.backward()                                # backpropagation
print("PyTorch: dL/dw =", w_t.grad.item())    # -16, same as by hand

### The gradient for softmax + cross-entropy
For our classifier there's a beautifully simple result: the gradient of the loss with respect to the scores is
$$\frac{\partial L}{\partial s} = P - \text{onehot}(y)$$
i.e. the probabilities, minus 1 at the correct class. Let's check it against PyTorch:

In [ ]:
s = torch.tensor(scores.ravel(), requires_grad=True)
loss_t = -torch.log(torch.softmax(s, dim=0)[correct])
loss_t.backward()

by_formula = prob.ravel().copy()
by_formula[correct] -= 1
print("PyTorch :", np.round(s.grad.numpy(), 4))
print("formula :", np.round(by_formula, 4))

## 5. Putting it together: a full training pipeline in NumPy
Below is a complete classifier trained **by hand** on a toy problem: 2-D points in 3 coloured clusters. It uses everything above: scores $= XW^T + b$, softmax, cross-entropy, the gradient formula, and gradient descent.

Notice the **structure** – this is what the Backprop lab asked you to identify, and Week 4 has exactly the same parts:

| Part | What it is | Week 4 PyTorch version |
|---|---|---|
| 1. Setup | imports, settings (learning rate, epochs) | `device`, `batch_size`, imports |
| 2. Data | get inputs `X` and labels `y`, split train/test | `datasets.MNIST`, `DataLoader` |
| 3. Model | the parameters and how they make scores | `class NeuralNetwork(nn.Module)` |
| 4. Loss & optimiser | how wrong + how to update | `nn.CrossEntropyLoss()`, `optim.SGD` |
| 5. Training loop | forward → loss → gradient → update, repeat | `train(...)` |
| 6. Evaluation | accuracy on held-out test data | `evaluate(...)` |

In [ ]:
# ---------- 1. Setup ----------
np.random.seed(0)
lr, epochs = 0.5, 200
num_classes = 3

# ---------- 2. Data ----------
centres = np.array([[0, 2], [-2, -1], [2, -1]])
X = np.vstack([np.random.randn(100, 2) * 0.8 + c for c in centres])   # 300 points, shape (300, 2)
y = np.repeat([0, 1, 2], 100)                                          # labels, shape (300,)
order = np.random.permutation(300)                                     # shuffle
X, y = X[order], y[order]
X_train, y_train, X_test, y_test = X[:240], y[:240], X[240:], y[240:]  # 80/20 split

# ---------- 3. Model ----------
W = np.random.randn(num_classes, 2) * 0.01      # (3 classes, 2 inputs)
b = np.zeros(num_classes)

def forward(X):
    S = X @ W.T + b                              # scores (N, 3) – same as Wx+b for every row
    S = S - S.max(axis=1, keepdims=True)         # stability trick: doesn't change softmax
    P = np.exp(S) / np.exp(S).sum(axis=1, keepdims=True)
    return P

# ---------- 4 & 5. Loss + training loop ----------
losses = []
for epoch in range(epochs):
    P = forward(X_train)                                         # forward pass
    N = len(y_train)
    loss = -np.mean(np.log(P[np.arange(N), y_train]))            # average cross-entropy
    losses.append(loss)

    dS = P.copy(); dS[np.arange(N), y_train] -= 1; dS /= N       # gradient wrt scores: P - onehot
    dW = dS.T @ X_train                                          # chain rule back to W
    db = dS.sum(axis=0)                                          # and to b

    W -= lr * dW                                                 # gradient descent step
    b -= lr * db

# ---------- 6. Evaluation ----------
def accuracy(X, y):
    return np.mean(np.argmax(forward(X), axis=1) == y)

print(f"start loss {losses[0]:.3f} (ln 3 = {np.log(3):.3f})  ->  final loss {losses[-1]:.3f}")
print(f"train accuracy {accuracy(X_train, y_train):.2%} | test accuracy {accuracy(X_test, y_test):.2%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(losses); ax1.set_title("training loss"); ax1.set_xlabel("epoch"); ax1.grid()

gx, gy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
regions = np.argmax(forward(np.c_[gx.ravel(), gy.ravel()]), axis=1).reshape(gx.shape)
ax2.contourf(gx, gy, regions, alpha=0.25, cmap="brg")
ax2.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="brg", edgecolor="k")
ax2.set_title("what the model learned (test points)")
plt.show()

### When would you change each part? (the Backprop lab's discussion question)
* **Data** – a new problem/dataset, or adding preprocessing (normalising, augmenting).
* **Model** – the model underfits (too simple: add layers/neurons) or overfits (too complex: make it smaller, add regularisation).
* **Loss** – a different kind of task (regression uses squared error; classification uses cross-entropy).
* **Optimiser / learning rate** – loss goes down too slowly (raise lr, try Adam) or jumps around/explodes (lower lr).
* **Training loop** – train longer (more epochs), use mini-batches for big data.
* **Evaluation** – always on data the model wasn't trained on.

Compare with the official PyTorch quickstart the lab links to (pytorch.org/tutorials/beginner/basics/intro.html) – you'll see the same six parts.

## Week 3 checklist
- I can compute $Wx + b$ and explain the shapes.
- I can compute hinge loss, softmax and $-\ln P$, and say what each is for.
- I know a random 10-class classifier has loss ≈ 2.3.
- I understand gradient descent ($w \leftarrow w - lr \cdot \text{grad}$) and that `backward()` = chain rule.
- I can name the six parts of an ML pipeline.

Next: **Week 4** – the same pipeline in PyTorch, on real MNIST images.